In [ ]:
import cv2 # pip install opencv-python + pip install opencv-contrib-python
import glob
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
from fcmeans import FCM
from PIL import Image
import time

In [ ]:
# ONLY NOISE APPLIED TO ALBEDO IMAGES WITH METHODS OF BELL AND CAREAGA

folder = "albedo_Bell"

files = [folder+"/albedo_papers_colors_v3_small_scene1_0347.png", folder+"/albedo_papers_colors_v3_small_scene2_0354.png", folder+"/albedo_papers_colors_v3_small_scene3_0364.png", folder+"/albedo_papers_colors_v3_small_scene4_0371.png"]

colors = ([420, 850], [550, 850], [690, 850], [850, 850])

df = pd.DataFrame() 
idx = 0

for f in files:
    
    print (f)
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    
    for c in colors:
        
        t_inici = time.time()
        
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    #index = h_roi.index(i)
                    #print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        #h_min = np.min(h_no_out)
        #h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)

        s_min = np.min(s_roi)
        s_max = np.max(s_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h_min, s_min, 0) 
        c2 = (h_max, s_max, 100)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-15:-4]+"_noise_Bell_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-15:-4]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        t_final = time.time()
        print ("Time: ", t_final-t_inici)

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Bell_results.xlsx") 

In [ ]:
# ONLY NOISE APPLIED TO ALBEDO IMAGES WITH METHODS OF BELL AND CAREAGA

folder = "albedo_Careaga"

files = [folder+"/albedo_papers_colors_v3_small_scene1_0347.png", folder+"/albedo_papers_colors_v3_small_scene2_0354.png", folder+"/albedo_papers_colors_v3_small_scene3_0364.png", folder+"/albedo_papers_colors_v3_small_scene4_0371.png"]

colors = ([420, 850], [550, 850], [690, 850], [850, 850])

df = pd.DataFrame() 
idx = 0

for f in files:
    
    print (f)
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    
    for c in colors:
        
        t_inici = time.time()
        
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    #index = h_roi.index(i)
                    #print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        #h_min = np.min(h_no_out)
        #h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)

        s_min = np.min(s_roi)
        s_max = np.max(s_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h_min, s_min, 0) 
        c2 = (h_max, s_max, 100)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-15:-4]+"_noise_Bell_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-15:-4]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        t_final = time.time()
        print ("Time: ", t_final-t_inici)

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Careaga_results.xlsx") 